In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
data_path = 'E:\\study\\Engineering\\FYP\\dataset\\archive\\*.csv'
files = glob.glob(data_path)

print(f"Found {len(files)} files.")

Found 10 files.


In [4]:
# Get feature names from first file (exclude non-features)
non_features = ['Flow Duration', 'Timestamp', 'Dst Port', 'Protocol', 'Label']
header = pd.read_csv(files[0], nrows=1)
feature_names = header.columns.drop(non_features)
num_features = len(feature_names)

In [5]:
# Initialize aggregators for all methods
corr_agg = np.zeros(num_features)  # For correlation
mi_agg = np.zeros(num_features)    # For mutual info
imp_agg = np.zeros(num_features)   # For RF importances
rfe_ranking_agg = np.zeros(num_features)  # For RFE rankings (lower is better)

In [7]:
for file in files:
    # Load in chunks to save memory
    chunks = pd.read_csv(file, chunksize=200000)  # Adjust chunk size based on RAM
    df_list = []
    for chunk in chunks:

        # Data cleaning
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.dropna(inplace=True)

        # Encode labels
        le = LabelEncoder()
        chunk['Label'] = chunk['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

        X_chunk = chunk.drop(non_features, axis=1)
        y_chunk = chunk['Label']

        # Standardize features
        scaler = StandardScaler()
        X_chunk_scaled = scaler.fit_transform(X_chunk)
        df_list.append((X_chunk_scaled, y_chunk))

    # Concat chunk data for this file
    X_file = np.vstack([x for x, _ in df_list])
    y_file = np.concatenate([y for _, y in df_list])

    # 1. Correlation Analysis (Pearson with target)
    corr_matrix = np.corrcoef(X_file.T, y_file)
    corr = np.abs(corr_matrix[:-1, -1])  # Absolute correlation with y
    corr_agg += corr

    # 2. Mutual Information
    mi = mutual_info_classif(X_file, y_file)
    mi_agg += mi

    # Subsample for model-based methods (50% to save time; increase if possible)
    X_train, _, y_train, _ = train_test_split(X_file, y_file, test_size=0.5, random_state=42)

    # 3. Random Forest Feature Importances
    rf = RandomForestClassifier(n_estimators=50, random_state=42)
    rf.fit(X_train, y_train)
    imp_agg += rf.feature_importances_

    # 4. Recursive Feature Elimination (RFE) - uses RF as estimator
    rfe = RFE(estimator=RandomForestClassifier(n_estimators=20, random_state=42),  # Fewer estimators for speed
              n_features_to_select=20, step=0.1)  # Select top 20, remove 10% per step
    rfe.fit(X_train, y_train)
    rfe_ranking_agg += rfe.ranking_  # Aggregate rankings (1 is best)

c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\vithu\AppData\Local\Temp\ipykernel_3916\2310876048.py:5: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,5

ValueError: could not convert string to float: 'Tot Fwd Pkts'

In [ ]:
# Average across files
num_files = len(files)
corr_avg = corr_agg / num_files
mi_avg = mi_agg / num_files
imp_avg = imp_agg / num_files
rfe_ranking_avg = rfe_ranking_agg / num_files

# Get top 20 for each (for RFE, sort by lowest average ranking)
top_corr = pd.Series(corr_avg, index=feature_names).sort_values(ascending=False).head(20)
top_mi = pd.Series(mi_avg, index=feature_names).sort_values(ascending=False).head(20
                                                                                  )
top_imp = pd.Series(imp_avg, index=feature_names).sort_values(ascending=False).head(20)
top_rfe = pd.Series(rfe_ranking_avg, index=feature_names).sort_values(ascending=True).head(20)  # Lower ranking better

In [ ]:
# === AFTER your loop and averaging ===

from sklearn.preprocessing import MinMaxScaler
from collections import Counter

scaler = MinMaxScaler()

# Raw scores
corr_scores = pd.Series(corr_avg, index=feature_names)
mi_scores = pd.Series(mi_avg, index=feature_names)
imp_scores = pd.Series(imp_avg, index=feature_names)
rfe_scores = 1 / pd.Series(rfe_ranking_avg, index=feature_names)

# Normalize
corr_norm = pd.Series(scaler.fit_transform(corr_scores.values.reshape(-1,1)).flatten(), index=feature_names)
mi_norm = pd.Series(scaler.fit_transform(mi_scores.values.reshape(-1,1)).flatten(), index=feature_names)
imp_norm = pd.Series(imp_scores, index=feature_names)
rfe_norm = pd.Series(scaler.fit_transform(rfe_scores.values.reshape(-1,1)).flatten(), index=feature_names)

# Aggregate
aggregate_score = (corr_norm + mi_norm + imp_norm + rfe_norm) / 4
best_features = aggregate_score.sort_values(ascending=False)

# Top 20 from each
top20_corr = set(top_corr.index)
top20_mi = set(top_mi.index)
top20_imp = set(top_imp.index)
top20_rfe = set(top_rfe.index)

all_top = list(top20_corr) + list(top20_mi) + list(top20_imp) + list(top20_rfe)
frequency = Counter(all_top)

# Final selection
final_selected = [f for f in best_features.head(20).index if frequency.get(f, 0) >= 3]

# === OUTPUT ===
print("\n" + "="*60)
print("FINAL FEATURE SELECTION RESULTS")
print("="*60)
print(f"Top 20 Aggregated Features:\n{best_features.head(20)}")
print(f"\nFeatures in 3+ methods: {dict({k: v for k, v in frequency.items() if v >= 3})}")
print(f"\nFINAL SELECTED FEATURES ({len(final_selected)}):")
print(final_selected)
print("="*60)